# CodeModeBench on a Kaggle GPU

Runs an experiment config (default: `configs/full.yaml`) with Ollama on a free Kaggle GPU, using the same CLI as local runs.

**Before running:** in *Settings*, enable **GPU** (T4 or P100) and **Internet**.

Runs are resumable. If a session ends, start a new one, re-upload `results.zip` from the previous session (cell 6) and run again: completed runs are skipped. Use `SHARD` to split a large grid across sessions.

In [ ]:
# 1. Settings
REPO_URL = "https://github.com/akashsreedhar/codemode-bench.git"  # your fork or the upstream repository
CONFIG = "configs/full.yaml"
MODELS = ["qwen3:8b", "qwen3:14b"]       # Ollama models to pull (must match the config)
MODEL_IDS = "qwen3-8b,qwen3-14b"          # config model ids to run in this session
SHARD = ""                                # e.g. "1/3" to run one third of the grid
PREVIOUS_RESULTS = "/kaggle/input/codemode-results/results.zip"  # optional: resume from an earlier session

In [ ]:
# 2. System dependencies: Node.js 22 and Ollama
!nvidia-smi --query-gpu=name,memory.total --format=csv
!curl -fsSL https://deb.nodesource.com/setup_22.x | bash - > /dev/null && apt-get install -y nodejs > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh > /dev/null
!node --version && ollama --version

In [ ]:
# 3. Start the Ollama server in the background and pull the models
import os, subprocess, time
env = dict(os.environ, OLLAMA_FLASH_ATTENTION="1", OLLAMA_KEEP_ALIVE="30m")
server = subprocess.Popen(["ollama", "serve"], env=env, stdout=open("/kaggle/working/ollama.log", "w"), stderr=subprocess.STDOUT)
time.sleep(5)
for m in MODELS:
    subprocess.run(["ollama", "pull", m], check=True)
!ollama list

In [ ]:
# 4. Get the benchmark and restore earlier results, if any
%cd /kaggle/working
!test -d codemode-bench || git clone --depth 1 {REPO_URL} codemode-bench
%cd /kaggle/working/codemode-bench
!npm ci --no-audit --no-fund
if os.path.exists(PREVIOUS_RESULTS):
    !unzip -o -q {PREVIOUS_RESULTS} -d .
!npx tsx src/cli.ts bench --config {CONFIG} --models {MODEL_IDS} {('--shard ' + SHARD) if SHARD else ''} --dry-run

In [ ]:
# 5. Run (resumable)
!npx tsx src/cli.ts bench --config {CONFIG} --models {MODEL_IDS} {('--shard ' + SHARD) if SHARD else ''}

In [ ]:
# 6. Package results for download (and for resuming in a later session)
!zip -q -r /kaggle/working/results.zip results .cache/llm
!npx tsx src/cli.ts report --no-readme --out /kaggle/working/report
print("Download /kaggle/working/results.zip and /kaggle/working/report/index.html from the Output tab.")